#主成分分析により多重共線性の解消を目指す

In [ ]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰
from sklearn.tree import DecisionTreeRegressor  #回帰木

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [ ]:
#データフレームの読み込み
'''
df1 VIFの高さを解消するために一部の列を消去する前のデータフレーム
df2 〃した後のデータフレーム

以降は基本的にdf2を用いて分析を行う。
sc_x,df_yはdf2を基に作成する。
ただし、必要があればdf1も利用する。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')
df2 = pd.read_csv('datafiles/df2_after_drop.csv')

sc_x = df2.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df2['SalePrice'])

In [ ]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
#各種モデルの作成

#重回帰
model1 = LinearRegression()
model1.fit(sc_x, df_y)

#リッジ回帰
model2 = Ridge(alpha = 100)
model2.fit(sc_x, df_y)

#ラッソ回帰
model3 = Lasso(alpha = 100)
model3.fit(sc_x, df_y)

#回帰木
model4 = DecisionTreeRegressor(max_depth = 10, random_state = 0)
model4.fit(sc_x, df_y)

In [ ]:
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

In [ ]:
#主成分分析にて多重共線性の解消を目指す
df1 = pd.read_csv('datafiles/df2_after_drop.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df1['SalePrice'])

In [ ]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

In [ ]:
sc_x.columns

In [ ]:
#GarageFinish列をダミー変数化した列一覧をリスト化し、主成分分析により一列化
GarageFinish_cols = []
for c in sc_x.columns:
    if 'GarageFinish_' in c:
        GarageFinish_cols.append(c)
print(GarageFinish_cols)


In [ ]:

PCAmodel = PCA(whiten = True)
GarageFinish_df = pd.DataFrame()
for c in GarageFinish_cols:
    GarageFinish_df = pd.concat([GarageFinish_df, sc_x[c]], axis = 1)
PCAmodel.fit(GarageFinish_df)
GarageFinish = PCAmodel.transform(sc_x[GarageFinish_cols])

#累積寄与率の閾値を0.8として、n_componentsを設定
thred = 0.8
final_num = 0
ratio =PCAmodel.explained_variance_ratio_
array = []
for i in range(len(ratio)):
    ruiseki = sum(ratio[0:i+1])    #i+1個めの特徴量までの累積寄与率
    if ruiseki > thred:   #i+1個めの特徴量において初めて累積寄与率がthredを超えるならば
          final_num = i
          break
PCAmodel = PCA(n_components = final_num, whiten = True)
PCAmodel.fit(sc_x[GarageFinish_cols])
GarageFinish = PCAmodel.transform(sc_x[GarageFinish_cols])
GarageFinish = pd.DataFrame(GarageFinish)

for c in sc_x[GarageFinish_cols]:
    sc_x = sc_x.drop([c], axis = 1)
sc_x = pd.concat([sc_x, GarageFinish], axis = 1)





result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


In [ ]:
#主成分分析で列を減らす（多重共線性の解消も目指す）